# 01 · Dal modello all'agente

Costruiamo, un pezzo alla volta, la scala che porta da una semplice chiamata al modello
fino a un **agente** che decide da solo quando usare uno strumento.

Tappe:
1. chiamata diretta al modello;
2. una *chain* (catena) con prompt e parser;
3. un **agente** con un tool e il suo loop ReAct;
4. ispezione del loop per vedere cosa succede davvero.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

## 1 · Chiamata diretta

Il livello più basso: mandiamo dei messaggi e riceviamo una risposta. Nessun prompt
riutilizzabile, nessuno strumento, nessun ciclo. Solo input → output.

In [ ]:
# I messaggi sono tipizzati: uno di "sistema" (le istruzioni) e uno "umano" (la domanda).
from langchain_core.messages import HumanMessage, SystemMessage

messaggi = [
    SystemMessage(content="Sei un docente di sistemi agentici. Rispondi in italiano, breve."),
    HumanMessage(content="In una frase: che differenza c'è tra un modello e un agente?"),
]

In [ ]:
# `invoke` fa una singola chiamata e restituisce un AIMessage.
risposta = model.invoke(messaggi)
print(risposta.text)   # `.text` estrae solo il testo della risposta

La risposta è un oggetto `AIMessage`. Contiene il testo ma anche metadati utili, come il
conteggio dei token. Diamoci un'occhiata.

In [ ]:
print("Tipo:", type(risposta).__name__)
print("Token usati:", risposta.usage_metadata)   # input/output/totale

## 2 · Una chain con LCEL

Una *chain* mette in fila più pezzi con l'operatore `|` (LangChain Expression Language).
Qui: un **prompt** con un segnaposto → il **modello** → un **parser** che tiene solo la stringa.

In [ ]:
# Il prompt ha una variabile {argomento}: potremo riusarlo con input diversi.
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sei un tutor di Python. Spiega con parole semplici e un mini esempio."),
    ("human", "Spiega: {argomento}"),
])

In [ ]:
# Il parser trasforma l'AIMessage in una semplice stringa.
from langchain_core.output_parsers import StrOutputParser

# `|` compone: prompt -> model -> parser. Il risultato è ancora un oggetto "eseguibile".
chain = prompt | model | StrOutputParser()

In [ ]:
# Invochiamo la chain riempiendo il segnaposto {argomento}.
print(chain.invoke({"argomento": "una list comprehension"}))

## 3 · Un agente con un tool

Un **agente** è un modello che può chiamare strumenti in un ciclo, finché non ha finito.
Trasformiamo una funzione Python in tool con `@tool`: i *tipi* e la *docstring* diventano
lo "schema" che il modello legge per capire quando e come usarlo.

In [ ]:
# La docstring spiega al modello COSA fa il tool; i tipi dicono quali argomenti servono.
from langchain_core.tools import tool


@tool
def area_rettangolo(base: float, altezza: float) -> float:
    """Calcola l'area di un rettangolo date base e altezza in metri."""
    if base <= 0 or altezza <= 0:
        raise ValueError("Le misure devono essere positive.")
    return base * altezza

In [ ]:
# `create_agent` costruisce il loop modello -> tool -> osservazione -> modello (pattern ReAct).
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[area_rettangolo],
    system_prompt="Usa sempre il tool per i calcoli, non stimare a mente.",
)

In [ ]:
# L'agente riceve i messaggi in un dizionario e restituisce lo stato finale.
esito = agente.invoke({
    "messages": [{"role": "user", "content": "Quanto misura l'area di 12.5 m per 8 m?"}]
})
print(esito["messages"][-1].text)   # l'ultimo messaggio è la risposta finale

## 4 · Guardare dentro il loop

La risposta finale nasconde i passaggi. La lista `messages` invece li mostra tutti:
la richiesta del tool da parte del modello, il risultato del tool, e la conclusione.

In [ ]:
# Stampiamo la sequenza di messaggi per vedere il ragionamento in azione.
for i, m in enumerate(esito["messages"]):
    tipo = type(m).__name__
    if getattr(m, "tool_calls", None):        # il modello ha CHIESTO di usare un tool
        print(f"{i}. {tipo}: chiama {m.tool_calls[0]['name']} con {m.tool_calls[0]['args']}")
    elif tipo == "ToolMessage":               # il RISULTATO del tool
        print(f"{i}. {tipo}: risultato = {m.content}")
    else:
        print(f"{i}. {tipo}: {str(m.content)[:80]}")

## Prova tu

- Aggiungi un tool `perimetro_rettangolo` e chiedi entrambi i valori.
- Rendi vaga la docstring del tool e osserva come il modello sbaglia a sceglierlo.

**Idea chiave**: una chain segue un percorso fisso deciso da te; un agente sceglie il
prossimo passo da solo, entro i limiti (i tool) che gli dai.